In [ ]:
!pip install pydicom
!pip install wget
!pip install open_clip_torch
!pip install --upgrade transformers

In [ ]:
from modules.dataset.factory import DatasetFactory
from modules.utils.constants import MODEL_TRANSFORMS, DEFAULT_TEMPLATES, RSNA_CLASS_PROMPTS, RSNA_CLASS_PROMPTS, SIZE_TRANSFORM, TENSOR_NORMALIZE_TRANSFORM,DATA_ROOT, ENTREP_CLASS_PROMPTS
from modules.models.factory import ModelFactory
from modules.utils.helpers import _extract_label, load_open_clip_model
import torch.nn.functional as F


In [ ]:
from torch.utils.data import Dataset
from torchvision import transforms
import csv
import os
from PIL import Image
_toTensor = transforms.ToTensor()
from torch.utils.data import Dataset

class MiMic(Dataset):
    def __init__(self, size_transform):
        from datasets import load_dataset
        self.ds = load_dataset(
            "itsanmolgupta/mimic-cxr-dataset",
            split="train",
            streaming=False
        )

        self.size_transform = size_transform
        self.samples = []

        for i in range(len(self.ds)):
            if self.ds[i]["findings"]:
                self.samples.append((i, "findings"))
            if self.ds[i]["impression"]:
                self.samples.append((i, "impression"))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ds_idx, field = self.samples[idx]

        sample = self.ds[ds_idx]
        img = sample["image"]          # load tại đây
        text = sample[field]

        img = self.size_transform(img).convert("RGB")
        img = _toTensor(img)

        return {
            "image": img,
            "text": text,
            "idx": idx
        }



class ENTREPDataset(Dataset):
    def __init__(
        self,
        size_transform,
        img_dir="/kaggle/input/entrep-train-contrastive/Dataset/images",
        annotation_file="/kaggle/input/entrep-train-contrastive/Dataset/data.csv",
    ):
        self.img_dir = img_dir
        self.annotation_file = annotation_file
        self.size_transform = size_transform

        self.samples = []

        with open(self.annotation_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                path = row["path"].replace('image', "Image")
                caption = row["caption"]

                img_path = os.path.join(self.img_dir, path)
                self.samples.append((img_path, caption))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]

        img = self.size_transform(Image.open(img_path)).convert("RGB")
        img = _toTensor(img)

        return {
            "image": img,
            "text": text,
            "idx": idx
        }



# Train Self-supervised Vision Encoder

In [ ]:
import os
import copy
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [ ]:
def ssl_infonce(z_q, z_k, temperature=0.07):
    logits = z_q @ z_k.t() / temperature
    labels = torch.arange(z_q.size(0), device=z_q.device)

    return F.cross_entropy(logits, labels)


In [ ]:
def freeze_model(model):
    for p in model.parameters():
        p.requires_grad = False


def freeze_text_encoder(model):
    if hasattr(model, "text_model"):
        for p in model.text_model.parameters():
            p.requires_grad = False


In [ ]:
def pgd_attack_from_frozen_model(
    model_source,
    imgs,
    eps=0.03,
    alpha=0.01,
    steps=10,
):
    model_source.eval()

    delta = torch.zeros_like(imgs).uniform_(-eps, eps).to(imgs.device)
    delta.requires_grad = True

    for _ in range(steps):
        adv_imgs = torch.clamp(imgs + delta, 0, 1)

        feats_adv = model_source.encode_posttransform_image(adv_imgs)
        feats_clean = model_source.encode_posttransform_image(imgs).detach()

        loss = ssl_infonce(feats_adv, feats_clean)
        loss.backward()

        delta.data = delta + alpha * delta.grad.sign()
        delta.data = torch.clamp(delta.data, -eps, eps)
        delta.grad.zero_()

    return torch.clamp(imgs + delta.detach(), 0, 1)


In [ ]:
from torchvision.utils import save_image

def generate_and_save_adv_images(
    model_source,
    dataloader,
    save_dir,
    eps=0.03,
    pgd_steps=10,
):
    os.makedirs(save_dir, exist_ok=True)
    freeze_model(model_source)
    model_source.eval()

    global_idx = 0
    for batch in tqdm(dataloader, desc="Generating adversarial data"):
        imgs = batch["image"].cuda()

        adv_imgs = pgd_attack_from_frozen_model(
            model_source,
            imgs,
            eps=eps,
            alpha=0.01,
            steps=pgd_steps,
        )

        for j in range(len(adv_imgs)):
            save_image(
                adv_imgs[j].cpu(),                 # (C,H,W), value in [0,1]
                os.path.join(save_dir, f"{global_idx}.png")
            )
            global_idx += 1



In [ ]:
class AdvDataset(Dataset):
    def __init__(self, size_transform, base_dataset, adv_dir):
        self.base_dataset = base_dataset
        self.adv_dir = adv_dir
        self.size_transform = size_transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        sample = self.base_dataset[idx]
        adv_img_path = torch.load(
            os.path.join(self.adv_dir, f"{idx}.png")
        )

        adv_img_path = self.size_transform(Image.open(adv_img_path)).convert("RGB")
        adv_img_path = _toTensor(adv_img_path)


        return {
            "image": adv_img,
            "text": sample["text"]
        }


In [ ]:
class AT_Dataset(Dataset):
    def __init__(self, dataset, adv_dataset):
        self.dataset = dataset
        self.adv_dataset = adv_dataset
    def __len__(self):
        return len(self.dataset)
    def __getitem__(self, idx):
        clean_sample = self.dataset[idx]
        adv_sample = self.dataset[idx]
        if clean_sample['text'] != adv_sample['text']:
            raise
            
        return {
            'image': clean_sample['image'],
            'adv_image': adv_sample['image'],
            'text': clean_sample['text']
        } 

In [ ]:
def train_on_adv_dataset(
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=10,
):
    model.train()
    freeze_text_encoder(model)

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            imgs = batch["image"].to(device)
            adv_imgs = batch["adv_image"].to(device)
            texts = batch["text"]

            z_adv = model.encode_posttransform_image(adv_imgs)
            z_clean = model.encode_posttransform_image(imgs).detach()

            loss = ssl_infonce(z_adv, z_clean)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[Epoch {epoch+1}] Loss: {total_loss/len(dataloader):.4f}")


# Config

In [ ]:
model_name = 'biomedclip'
dataset_name = 'mimic'
mode_pretrained = 'scratch'
size_transform = SIZE_TRANSFORM[model_name]
batch_size = 64
epochs = 50
epsilon = 0.03

In [ ]:
from torch.utils.data import DataLoader
import torch
import yaml

if model_name in ['medclip', 'biomedclip']:
    model = ModelFactory.create_model(
        model_type=model_name,
        variant='base',
        pretrained=True,
        mode_pretrained=mode_pretrained
    )

elif model_name == 'entrep':
    config_path = "configs/entrep_contrastive.yaml"
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    model_config = config.get('model', {})
    model = ModelFactory.create_model(
        model_type="entrep",
        variant='base',
        checkpoint=None,
        pretrained=False,
        **{k: v for k, v in model_config.items() if k != 'model_type' and k != "pretrained" and k != "checkpoint"},
        mode_pretrained=mode_pretrained
    )   

import copy

model = model.cuda()

# ===== create frozen source model for PGD =====
model_source = copy.deepcopy(model).cuda()
model_source.eval()


if dataset_name == "mimic":
    dataset = MiMic(
        size_transform=size_transform,
    )
elif dataset_name == "entrep":
    dataset = ENTREPDataset(
        size_transform=size_transform
    )


gen_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,   # 🔥 FIX
)

generate_and_save_adv_images(
    model_source,
    gen_loader,
    save_dir="adv_eps003_steps10",
    eps=0.03,
    pgd_steps=10,
)

# ===== adversarial dataset =====
adv_dataset = AdvDataset(size_transform, dataset, adv_dir="adv_eps003_steps10")
AT_dataset = AT_Dataset(dataset, adv_dataset)

AT_loader = DataLoader(
    AT_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

if model_name == "biomedclip":
    # ===== optimizer (vision only) =====
    optimizer = torch.optim.AdamW(
        model.visual.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )


else:
    # ===== optimizer (vision only) =====
    optimizer = torch.optim.AdamW(
        model.vision_model.parameters(),
        lr=1e-4,
        weight_decay=1e-4
    )



In [ ]:
train_on_adv_dataset(
    model,
    AT_loader,
    optimizer,
    epochs=epochs
)

# Contasive Learning

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

def clip_infonce_loss(image_feats, text_feats, temperature=0.07):

    logits = image_feats @ text_feats.t() / temperature
    labels = torch.arange(logits.size(0), device=logits.device)

    loss_t2i = F.cross_entropy(logits.t(), labels)
    loss_i2t = F.cross_entropy(logits, labels)

    return 0.5 * (loss_t2i + loss_i2t)


In [ ]:
def train_clip(
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=10,
    temperature=0.07,
):
    model.train()

    for epoch in range(epochs):
        total_loss = 0.0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
            imgs  = batch["image"].to(device)
            texts = batch["text"]

            # ===== forward =====
            image_feats = model.encode_posttransform_image(imgs)
            text_feats  = model.encode_text(texts)

            loss = clip_infonce_loss(
                image_feats,
                text_feats,
                temperature=temperature
            )

            # ===== backward =====
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(
            f"[Epoch {epoch+1}/{epochs}] "
            f"CLIP Loss: {total_loss/len(dataloader):.4f}"
        )


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

dataloader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    drop_last=True
)

# ===== optimizer =====
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ===== train =====
train_clip(
    model,
    dataloader,
    optimizer,
    device="cuda",
    epochs=epochs,
    temperature=0.07
)


In [ ]:
torch.save(model.state_dict(), f'{model_name}_AT.pth')